# 🔍 สำรวจข้อมูล + ออกแบบ Star Schema

เป้าหมาย: **เข้าใจข้อมูล + เห็นปัญหาที่ต้อง clean + รู้ว่าจะปั้นเป็น star schema หน้าตาไหน**

## 1) โหลดข้อมูล

In [1]:
import pandas as pd
df = pd.read_csv("../data/loan_sample.csv")

print("ขนาด:", df.shape)
#df.head()
df.tail(10)

ขนาด: (5020, 22)


,id,loan_amnt,funded_amnt,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,...,issue_d,loan_status,purpose,addr_state,dti,fico_range_low,fico_range_high,total_pymnt,total_rec_prncp,recoveries
5010,9000010,5000,0,36 months,10.56%,162.65,B,B2,7 years,OWN,...,Apr-2017,Charged Off,other,IL,6.40,760,764,2735.58,2188.47,419.01
5011,9000011,10000,0,60 months,16.22%,244.35,C,C5,7 years,RENT,...,Jan-2018,Fully Paid,debt_consolidation,PA,14.50,780,784,14379.70,10000.00,0.00
5012,9000012,25000,0,60 months,31.07%,825.34,G,G1,1 year,OWN,...,May-2016,Charged Off,house,VA,5.87,660,664,8333.89,6667.11,901.54
5013,9000013,20000,0,60 months,22.72%,560.60,E,E1,10+ years,MORTGAGE,...,Mar-2016,Current,credit_card,NC,21.99,720,724,4694.58,2791.40,0.00
5014,9000014,35000,0,36 months,6.61%,1074.47,A,A3,< 1 year,RENT,...,Mar-2016,Fully Paid,small_business,PA,34.30,660,664,36896.36,35000.00,0.00
5015,9000015,8000,8000,36 months,6.49%,245.16,A,A2,10+ years,MORTGAGE,...,Feb-2016,In Grace Period,credit_card,GA,31.01,780,784,4093.45,3710.46,0.00
5016,9000016,12000,12000,36 months,14.71%,414.28,C,C2,10+ years,OWN,...,Apr-2017,Fully Paid,other,WA,34.59,780,784,14367.48,12000.00,0.00
5017,9000017,8000,8000,36 months,6.43%,244.94,A,A5,5 years,RENT,...,Feb-2018,Fully Paid,vacation,AZ,24.61,675,679,8802.31,8000.00,0.00
5018,9000018,20000,20000,60 months,11.07%,435.55,B,B1,< 1 year,OWN,...,Apr-2016,Charged Off,other,NC,2.18,660,664,3744.81,2995.85,53.34
5019,9000019,12000,12000,36 months,20.62%,449.76,D,D3,10+ years,MORTGAGE,...,Aug-2016,Fully Paid,credit_card,VA,18.34,675,679,15530.87,12000.00,0.00


## 2) มีคอลัมน์อะไรบ้าง

In [2]:
df.columns

Index(['id', 'loan_amnt', 'funded_amnt', 'term', 'int_rate', 'installment',
       'grade', 'sub_grade', 'emp_length', 'home_ownership', 'annual_inc',
       'verification_status', 'issue_d', 'loan_status', 'purpose',
       'addr_state', 'dti', 'fico_range_low', 'fico_range_high', 'total_pymnt',
       'total_rec_prncp', 'recoveries'],
      dtype='str')

In [3]:
df.columns.tolist()

['id',
 'loan_amnt',
 'funded_amnt',
 'term',
 'int_rate',
 'installment',
 'grade',
 'sub_grade',
 'emp_length',
 'home_ownership',
 'annual_inc',
 'verification_status',
 'issue_d',
 'loan_status',
 'purpose',
 'addr_state',
 'dti',
 'fico_range_low',
 'fico_range_high',
 'total_pymnt',
 'total_rec_prncp',
 'recoveries']

## 3) 🚩 หา "ปัญหาข้อมูล" — ของจริงไม่เคยสะอาด

ลองดูคอลัมน์พวกนี้ — สังเกตว่าเก็บมาเป็น **ข้อความปนสัญลักษณ์** ใช้คำนวณไม่ได้ทันที

In [3]:
df[["int_rate", "term", "emp_length", "issue_d", "loan_status"]].head(8)

,int_rate,term,emp_length,issue_d,loan_status
0,15.37%,60 months,1 year,Sep-2016,Fully Paid
1,13.44%,36 months,< 1 year,Mar-2017,Fully Paid
2,12.06%,60 months,2 years,Jan-2016,Current
3,5.97%,36 months,7 years,Jul-2016,Fully Paid
4,18.78%,36 months,1 year,Jan-2018,In Grace Period
5,14.58%,36 months,1 year,Jul-2017,Current
6,17.57%,36 months,5 years,Feb-2017,Fully Paid
7,16.35%,36 months,5 years,Jan-2018,Fully Paid


เห็นปัญหาไหม?
- `int_rate` = `"13.56%"` → มี `%` (เป็นข้อความ ไม่ใช่ตัวเลข)
- `term` = `" 36 months"` → มีคำว่า months + เว้นวรรคหน้า
- `emp_length` = `"10+ years"` / `"< 1 year"` → ข้อความ
- `issue_d` = `"Dec-2018"` → ต้องแปลงเป็นวันที่

## 4) ดู type จริง + ค่าว่าง (null)

In [14]:
print("=== dtypes ===")
print(df.dtypes)
print("\n=== จำนวน null ต่อคอลัมน์ ===")
print(df.isnull().sum())

=== dtypes ===
id                       int64
loan_amnt                int64
funded_amnt              int64
term                       str
int_rate                   str
installment            float64
grade                      str
sub_grade                  str
emp_length                 str
home_ownership             str
annual_inc             float64
verification_status        str
issue_d                    str
loan_status                str
purpose                    str
addr_state                 str
dti                    float64
fico_range_low           int64
fico_range_high          int64
total_pymnt            float64
total_rec_prncp        float64
recoveries             float64
dtype: object

=== จำนวน null ต่อคอลัมน์ ===
id                       0
loan_amnt                0
funded_amnt              0
term                     0
int_rate                 0
installment              0
grade                    0
sub_grade                0
emp_length             629
home_ownership 

สังเกต: คอลัมน์ที่ควรเป็นตัวเลข/วันที่ กลับเป็น `object` (ข้อความ) → **นี่คืองานของ ETL ตอนต่อไป**

## 5) สถานะสินเชื่อมีอะไรบ้าง (สำคัญกับ default rate)

In [15]:
df["loan_status"].value_counts()

loan_status
Fully Paid            2256
Current               1233
Charged Off            929
Late (31-120 days)     259
In Grace Period        199
Default                139
UNKNOWN                  5
Name: count, dtype: int64

เราจะจัดกลุ่มสถานะเป็น **Good / Bad / In Progress** และทำ flag `is_default`
(เช่น Charged Off, Default = เบี้ยวหนี้) ในขั้น transform

## 6) 🎯 เป้าหมาย: Star Schema

**Grain = 1 แถวต่อ 1 สินเชื่อ** ตรงกลางคือ **fact** (ตัวเลขวัดผล) ล้อมด้วย **dimension** (มุมมอง)

```
        dim_date     dim_grade     dim_loan_status
            \          |             /
 dim_purpose ── ★ fact_loan ★ ── dim_borrower
            /          |             \
      dim_geography  dim_term     (loan_amnt, int_rate,
                                   is_default, profit ...)
```

| ตาราง | เก็บอะไร |
|---|---|
| `fact_loan` | 1 แถว/สินเชื่อ + ตัวเลข (วงเงิน, ดอกเบี้ย, is_default, profit) + คีย์ไปหา dim |
| `dim_date` | เดือนที่ปล่อยกู้ (issue_d) |
| `dim_grade` | เกรด A–G + sub-grade + ระดับความเสี่ยง |
| `dim_purpose` | วัตถุประสงค์ + หมวด |
| `dim_loan_status` | สถานะ + กลุ่ม (Good/Bad) + is_default |
| `dim_geography` | รัฐ + ภูมิภาค |
| `dim_borrower` | โปรไฟล์ผู้กู้ (อายุงาน, บ้าน, ช่วงรายได้) |
| `dim_term` | งวด 36 / 60 เดือน |

---
✅ เข้าใจข้อมูล + เห็นปัญหา + รู้เป้าหมาย Star Schema แล้ว
ไปต่อ **`02_etl.ipynb`** — ลงมือ **clean → validate → transform** ของจริง